# TemporalRec — Cleaning impact and repeat behavior

Run one cell at a time. This notebook answers:
1. Did cleaning remove purchases more often than views?
2. How often is the next retained event for the same item?

These are descriptive checks. They do not train a model or change the input files. Percentages describe our chosen data population, not all shoppers.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Works when Jupyter starts in the project root or its notebooks folder.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "dataset" / "events.csv").exists()), None
)
assert PROJECT_ROOT is not None, "Open this notebook inside the Recommendation project."
INPUT_RUN_TAG = "notebook_run_2026_10_13"
FIVE_CORE_RUN_TAG = "five_core_2026_10_13"
RAW_FILE = PROJECT_ROOT / "dataset/events.csv"
TIE_FREE_FILE = PROJECT_ROOT / "data/interim" / INPUT_RUN_TAG / "events_clean_for_next_steps.csv"
CORE_FILE = PROJECT_ROOT / "data/interim" / FIVE_CORE_RUN_TAG / "events_all_event_types_5core.csv"
for path in [RAW_FILE, TIE_FREE_FILE, CORE_FILE]:
    assert path.exists(), f"Missing input: {path}"

RUN_TAG = "additional_eda_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
REPORT_DIR = PROJECT_ROOT / "reports" / RUN_TAG
REPORT_DIR.mkdir(parents=True, exist_ok=False)
EVENT_TYPES = ["view", "addtocart", "transaction"]
FIELDS = ["timestamp", "visitorid", "event", "itemid", "transactionid"]
DTYPES = {c: "int64" for c in ["timestamp", "visitorid", "itemid"]}
DTYPES.update(event="string", transactionid="Int64")
print("Reports:", REPORT_DIR)

## 1. Reconstruct and verify the cleaning stages

We keep each row's original index, so index + 2 points to its row in the raw CSV (including the header).
We also compare the reconstructed data with your saved notebook outputs.

In [ ]:
raw = pd.read_csv(RAW_FILE, dtype=DTYPES)
assert raw.columns.tolist() == FIELDS
assert raw[["timestamp", "visitorid", "event", "itemid"]].notna().all().all()
assert set(raw["event"]) <= set(EVENT_TYPES)
deduped = raw.drop_duplicates(FIELDS, keep="first").copy()
tie_mask = deduped.duplicated(["visitorid", "timestamp"], keep=False)
tie_free = deduped.loc[~tie_mask].copy()
saved_tie_free = pd.read_csv(TIE_FREE_FILE, dtype=DTYPES)
pd.testing.assert_frame_equal(tie_free.reset_index(drop=True), saved_tie_free)
del saved_tie_free
core = pd.read_csv(CORE_FILE, dtype=DTYPES)
assert len(core) > 0, "The 5-core output is empty."
assert not core.duplicated(FIELDS).any()
assert not core.duplicated(["visitorid", "timestamp"]).any()
# Each saved core row must match a row in the preceding stage.
matched = core.merge(tie_free, on=FIELDS, how="left", indicator=True, validate="one_to_one")
assert matched["_merge"].eq("both").all()
del matched
assert core.groupby("visitorid").size().min() >= 5
assert core.groupby("itemid").size().min() >= 5
print("Saved inputs match the cleaning policy; core rows belong to the preceding stage.")

## 2. Count losses separately for each action

A small overall loss can hide a large purchase loss. Each loss percentage below uses the count of that **same event type before that step** as its denominator.
The final 5-core stage is shown separately because it selects active users and items.

In [ ]:
stages = {"raw": raw, "exact_deduplicated": deduped, "tie_free": tie_free, "five_core": core}
counts = pd.DataFrame({
    name: frame["event"].value_counts().reindex(EVENT_TYPES, fill_value=0)
    for name, frame in stages.items()
}).rename_axis("event")
counts.to_csv(REPORT_DIR / "01_event_counts_by_stage.csv")
loss_rows = []
for before, after in zip(list(stages)[:-1], list(stages)[1:]):
    for event in EVENT_TYPES:
        n_before, n_after = int(counts.loc[event, before]), int(counts.loc[event, after])
        assert n_after <= n_before
        loss_rows.append({
            "step": f"{before} -> {after}", "event": event,
            "before_count": n_before, "after_count": n_after,
            "removed_count": n_before - n_after,
            "removed_percent": 100 * (n_before - n_after) / n_before if n_before else np.nan,
        })
losses = pd.DataFrame(loss_rows)
losses.to_csv(REPORT_DIR / "02_event_losses.csv", index=False)
display(counts)
display(losses.round(3))
plot_data = losses.pivot(index="event", columns="step", values="removed_percent").reindex(EVENT_TYPES)
ax = plot_data.plot.bar(figsize=(11, 5))
ax.set(xlabel="Event type", ylabel="Removed in this step (%)", title="Cleaning and filtering impact by event type")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Step", fontsize=8)
plt.tight_layout()
plt.savefig(REPORT_DIR / "event_losses.png", dpi=160)
plt.show()

### Inspect the affected purchases

The next file contains every transaction event excluded by the timestamp-tie policy, with its raw source row number. Tied purchases can be real basket purchases. Their removal is a modeling choice, not proof of bad data.

If purchases lose a much larger percentage than views, reconsider the earlier claim that tie removal was the “safest” rule. Compare alternatives before finalizing evaluation.

In [ ]:
tied_purchases = deduped.loc[tie_mask & deduped["event"].eq("transaction")].copy()
tied_purchases.insert(0, "raw_source_row_number", tied_purchases.index + 2)
tied_purchases.to_csv(REPORT_DIR / "03_excluded_tied_purchases.csv", index=False)
display(tied_purchases.head(10))
print("Transaction events excluded by tie policy:", len(tied_purchases))

## 3. Consecutive same-item events

For each visitor, sort by time and compare each item with the previous item.
The first event has no previous event, so it is excluded from the denominator.

Example: A → A → B has two transitions and one same-item transition: 50%.
We compare the tie-free population with the 5-core population. Filtering can make previously separated events become neighbors.

In [ ]:
def transition_table(frame):
    ordered = frame.sort_values(["visitorid", "timestamp"], kind="stable").copy()
    grouped = ordered.groupby("visitorid", sort=False)
    ordered["previous_itemid"] = grouped["itemid"].shift().astype("Int64")
    ordered["previous_event"] = grouped["event"].shift()
    ordered["previous_timestamp"] = grouped["timestamp"].shift().astype("Int64")
    transitions = ordered.loc[ordered["previous_itemid"].notna()].copy()
    transitions["same_item"] = transitions["itemid"].eq(transitions["previous_itemid"]).astype(bool)
    transitions["gap_seconds"] = (transitions["timestamp"] - transitions["previous_timestamp"]) / 1000
    assert transitions["gap_seconds"].gt(0).all()
    assert len(transitions) == len(frame) - frame["visitorid"].nunique()
    return transitions

# Small example checks that first events and different visitors are handled correctly.
toy = pd.DataFrame({
    "visitorid": [1, 1, 1, 2], "timestamp": [1000, 2000, 3000, 1000],
    "itemid": [10, 10, 20, 10], "event": ["view"] * 4,
})
toy_transitions = transition_table(toy)
assert len(toy_transitions) == 2 and toy_transitions["same_item"].sum() == 1

repeat_rows = []
for name, frame in {"tie_free": tie_free, "five_core": core}.items():
    transitions = transition_table(frame)
    repeat_rows.append({
        "population": name, "transitions": len(transitions),
        "same_item_transitions": int(transitions["same_item"].sum()),
        "same_item_percent": 100 * transitions["same_item"].mean(),
        "median_same_item_gap_seconds": transitions.loc[transitions["same_item"], "gap_seconds"].median(),
    })
    if name == "five_core":
        core_transitions = transitions
repeat_summary = pd.DataFrame(repeat_rows)
repeat_summary.to_csv(REPORT_DIR / "04_repeat_summary.csv", index=False)
display(repeat_summary.round(3))

## 4. Which event transitions repeat the same item?

A view → view repeat and a view → cart repeat can mean different things.
The table reports a separate repeat percentage for each event-type combination.

In [ ]:
by_event = (
    core_transitions.groupby(["previous_event", "event"], observed=True)
    .agg(transitions=("same_item", "size"), same_item_transitions=("same_item", "sum"))
    .reset_index()
)
by_event["same_item_percent"] = 100 * by_event["same_item_transitions"] / by_event["transitions"]
assert by_event["transitions"].sum() == len(core_transitions)
by_event.to_csv(REPORT_DIR / "05_repeats_by_event_transition.csv", index=False)
display(by_event.round(3))

labels = by_event["previous_event"] + " -> " + by_event["event"]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(labels, by_event["same_item_percent"])
ax.set(xlabel="Same-item transitions (%)", ylabel="Previous event -> current event",
       xlim=(0, 100), title="Repeat rates within the 5-core population")
fig.tight_layout()
fig.savefig(REPORT_DIR / "repeat_rates.png", dpi=160)
plt.show()

# Deterministic examples for manual inspection, not a representative sample.
example_columns = ["visitorid", "previous_timestamp", "timestamp", "previous_itemid",
                   "itemid", "previous_event", "event", "same_item", "gap_seconds"]
examples = core_transitions.loc[core_transitions["same_item"], example_columns].head(200)
examples.to_csv(REPORT_DIR / "06_same_item_examples.csv", index=False)
display(examples.head(10))

## What these results mean

- A high repeat rate suggests a useful future baseline: recommend the visitor's last item.
  The repeat rate above is descriptive; it is **not** a held-out model score.
- Varied time gaps give a reason to test TiSASRec. They do not prove it is better.
- The dataset produced by whole-timeline 5-core filtering is an exploratory population.
  Future interactions affected which visitors and items survived.
- Before training, choose global time boundaries and define eligibility and the item catalog using training-period evidence. Handle new users/items explicitly.
  Do not call a later chronological split alone proof that the entire pipeline is free of future information.
- Use training data for model-design decisions and validation for tuning. Preserve a final test period for evaluation.

## Manual Excel checks

1. Open **01_event_counts_by_stage.csv**: check how transaction counts change across stages.
2. Open **02_event_losses.csv**: use `=(C2-D2)/C2*100` in a spare column.
   It should match `removed_percent` in column F (when C2 is nonzero).
3. Open **04_repeat_summary.csv**: use `=C2/B2*100`; compare with column D.
4. Open **06_same_item_examples.csv**: previous_itemid (column D) must equal itemid (column E).
   Use `=(C2-B2)/1000` to check gap_seconds in column I.
5. The example file shows the first 200 matching transitions. Use the full summary for conclusions.

Stop here after running. Review purchase losses and repeat rates before choosing any additional cleaning or split rules.